In [ ]:
# pip: pip install selenium webdriver-manager
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import time
import os

def collect_data(filename):

    log_file = open(f'collect_html_{filename}.log', 'w', encoding='utf8')

    def write_log(log):
        print(log)
        log_file.write(str(log))
        log_file.write('\n')    
        log_file.flush()

    with open(filename, 'r', encoding='utf8') as fp:
        names = fp.readlines()

    base_url = "https://ecolife.me.go.kr/ecolife/chemiProd/safeDclrProd?pMENU_NO=596"
    save_dir = 'htmls'
    os.makedirs(save_dir, exist_ok=True)

    collected_names = {}
    for root, dirs, files in os.walk(save_dir):
        for f in files:
            collected_name = f.replace('.html','').rsplit('+')[1]
            collected_names[collected_name] = True

    write_log(collected_names)

    # ==== Setup driver (Chrome) ====
    options = webdriver.ChromeOptions()

    # options.add_argument("--headless=new")  # uncomment to run headless
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) ...")

    driver = webdriver.Chrome(options=options)
    driver.maximize_window()
    wait = WebDriverWait(driver, 10)    

    def wait_for_overlay_to_disappear(driver, check_interval=1):
        write_log("Waiting for overlay to disappear...")
        while True:
            try:
                overlay = driver.find_element(By.CSS_SELECTOR, "div.blockUI.blockMsg.blockPage")
                if overlay.is_displayed():
                    time.sleep(check_interval)
                else:
                    break  # overlay not visible anymore
            except NoSuchElementException:
                break  # overlay not present at all
        write_log("Overlay disappeared.")


        
    for n in names:    
        name = n.strip()
        try:
            write_log(f'name: {name}') 

            if name in collected_names:
                write_log(f'{name} has been already collected')
                continue

            
            driver.get(base_url)

            # Wait for loading overlay to disappear
            wait_for_overlay_to_disappear(driver)

            search_input = wait.until(EC.visibility_of_element_located((By.ID, "pSearchWord")))               
            search_section = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.search-section")))
            search_button = search_section.find_element(By.CSS_SELECTOR, "button.btn-search")
            wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.search-section button.btn-search")))        
            search_input.clear()
            search_input.send_keys(name)     
            search_button.click()   

            wait_for_overlay_to_disappear(driver)

            # find all link elements on the page
            anchors = driver.find_elements(By.CSS_SELECTOR, "td.align-mo-l a")
            # iterate safely (copy list so DOM changes don't break iteration)
            anchors_snapshot = list(anchors)

            for idx, a in enumerate(anchors_snapshot):
                try:
                    # Find the parent <tr> of this <a>
                    tr = a.find_element(By.XPATH, "./ancestor::tr")

                    # Get all <td> in that <tr>
                    tds = tr.find_elements(By.TAG_NAME, "td")

                    # 신고번호 is in the 7th <td> (index 6)
                    report_no = tds[6].text.strip()

                    write_log(f'report_no: {report_no}')

                    # retrieve the onclick JS (e.g., "goView('NLC_0C7CCEEF5','15');")
                    onclick_js = a.get_attribute("onclick") or a.get_attribute("href")
                    if onclick_js and "goView" in onclick_js:
                        # execute JS directly (same effect as clicking)
                        driver.execute_script(onclick_js)
                    else:
                        # fallback: normal click
                        a.click()

                    # Wait for loading overlay to disappear
                    wait_for_overlay_to_disappear(driver)
                    
                    # Wait until the list is present
                    wait.until(EC.presence_of_element_located((By.ID, "excelTarget1")))

                    # sanitize filename (remove slashes, etc.)
                    safe_report_no = report_no.replace("/", "-").replace("\\", "-")
                    save_path = os.path.join(save_dir, f"{safe_report_no}+{name}.html")

                    # save full HTML
                    with open(save_path, "w", encoding="utf-8") as fp:
                        fp.write(driver.page_source)

                    log = f"Saved detail page as {save_path}"
                    write_log(log)

                    # go back to list page (or call history.goBack)
                    driver.back()
                    # wait until list re-appears
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "td.align-mo-l a")))
                    time.sleep(0.2)

                except Exception as e:
                    write_log(f'Failed at index {idx}')
                    break
        except:
            write_log(f'Failed at name {name}')
        


from concurrent.futures import ThreadPoolExecutor
import glob

def main():
    files = glob.glob("names_part_*.txt")  # or your actual split file pattern

    print(files)

    with ThreadPoolExecutor(max_workers=20) as executor:  # adjust number of workers as per your CPU/ram
        executor.map(collect_data, files)


main()


In [ ]:
def split_file_into_parts(input_file, parts=12, output_prefix="part"):
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    total_lines = len(lines)
    lines_per_part = total_lines // parts
    remainder = total_lines % parts

    start = 0
    for i in range(parts):
        # Distribute the remainder lines across the first 'remainder' parts
        end = start + lines_per_part + (1 if i < remainder else 0)
        part_lines = lines[start:end]

        output_file = f"{output_prefix}_{i+1}.txt"
        with open(output_file, "w", encoding="utf-8") as out_f:
            out_f.writelines(part_lines)

        print(f"Saved {output_file} with {len(part_lines)} lines.")
        start = end

# Usage example:
split_file_into_parts("names.txt", parts=5, output_prefix="names_part")
